In [1]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import shutil
from tqdm import tqdm, trange
from pathlib import Path
import ants
from nilearn.image import resample_to_img, load_img
import seaborn as sns
import tempfile

plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'Arial'


PLOTS_PATH = Path('../plots')
DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')

# Copy and Transform Nifti File

## BCP

In [2]:
for k_num in range(2, 7):
    input_file = DATA_PATH / f'cbptools/BCP_age_0_3_resting/K{k_num}.nii.gz'

    # Updated output file path as requested
    output_file = DATA_PATH / f'cbptools/BCP_age_0_3_resting_remap/K{k_num}.nii.gz'

    # step 1: baby template space
    baby_template_file = '/home/guoqiu/NAS/NAS4/Dep/BCP/BCP_from_Helab/UNC-BCP_4D_Infant_Brain_Volumetric_Atlas/6Month/BCP-06M-T1.nii.gz'
    baby_input_nib_img = resample_to_img(
        source_img=input_file,
        target_img=baby_template_file,
        interpolation='nearest')

    # Convert Nibabel to ANTsImage safely
    with tempfile.NamedTemporaryFile(suffix='.nii.gz') as tmp:
        nib.save(baby_input_nib_img, tmp.name)
        baby_ants_img = ants.image_read(tmp.name)

    # Continue with your existing pipeline
    MNI152_template_file = "/home/guoqiu/NAS/NAS2/Baby_Brain/dHCP/Infant_registration_2step_extdhcp40wk/mni_icbm152_t1_tal_nlin_asym_09a_brain.nii"
    MNI152_template_ants_img = ants.image_read(MNI152_template_file)

    transform_list = ants.registration(
        fixed=ants.image_read(baby_template_file),
        moving=MNI152_template_ants_img,
        type_of_transform='SyN')['invtransforms']

    MNI_input_ants_img = ants.apply_transforms(
        moving=baby_ants_img,
        fixed=MNI152_template_ants_img,
        interpolator='nearestNeighbor',
        transformlist=transform_list)

    # step 3: MNI152 to vmpfc
    vmpfc_mask_file = DATA_PATH / "K3_VMPFC_mask_binary.nii.gz"

    with tempfile.NamedTemporaryFile(suffix='.nii.gz') as tmp2:
        ants.image_write(MNI_input_ants_img, tmp2.name)
        temp_nib = nib.load(tmp2.name)
        MNI_input_nib_img_final = nib.Nifti1Image(temp_nib.get_fdata().copy(), temp_nib.affine, temp_nib.header)

    vmpfc_input_nib_img = resample_to_img(MNI_input_nib_img_final, vmpfc_mask_file, interpolation='nearest')

    vmpfc_input_nib_img.to_filename(output_file)
    print(f'from {input_file} \nto {output_file}')

from ../data/cbptools/BCP_age_0_3_resting/K2.nii.gz 
to ../data/cbptools/BCP_age_0_3_resting_remap/K2.nii.gz
from ../data/cbptools/BCP_age_0_3_resting/K3.nii.gz 
to ../data/cbptools/BCP_age_0_3_resting_remap/K3.nii.gz
from ../data/cbptools/BCP_age_0_3_resting/K4.nii.gz 
to ../data/cbptools/BCP_age_0_3_resting_remap/K4.nii.gz
from ../data/cbptools/BCP_age_0_3_resting/K5.nii.gz 
to ../data/cbptools/BCP_age_0_3_resting_remap/K5.nii.gz
from ../data/cbptools/BCP_age_0_3_resting/K6.nii.gz 
to ../data/cbptools/BCP_age_0_3_resting_remap/K6.nii.gz
